# demo

> Isolated rendering of markup with current styles and behaviors, for notebooks and docs

In [ ]:
#| default_exp demo

The registry holds styles and behaviors, and the behavior runtime turns them into JavaScript. To develop a component, we need to see the result and try its interactions. `demo` renders ordinary FastHTML markup in an isolated document with the current assets. The same preview is used both during notebook authoring and when viewing generated docs.


In [ ]:
from fastcore.utils import *
from fasthtml.common import to_xml, Div, Iframe, Button, Html, Head, Body, Meta, Style, Script
from fasthtml.core import htmxsrc
from answerui.registry import reg, Hyperelement
import answerui.behaviors  # installs Registry.build_js

In [ ]:
from answerui.registry import hypelt
from fastcore.test import *
from fasthtml.common import P, Details, Summary

## Rendering previews

A preview needs its own stylesheet and behavior script, as well as htmx. FastHTML exports its default script as `htmxsrc`, so we use that.

Both scripts load in the head, so the behavior listeners are installed before htmx fires `htmx:load` for the initial page. Supplying `css` replaces the registered stylesheet, which lets us experiment with styling independently of the rest of the library. The assets come from `reg` unless another registry is passed, so a registry of one's own previews the same way.

In [ ]:
def demo_html(
    *fts,         # Markup to render
    registry=reg, # Registry whose stylesheet and behavior script the document carries
    css=None,     # Stylesheet for the document; None uses the registry's build_css()
):
    "A standalone document containing markup, CSS, behaviors, and FastHTML's default htmx"
    head = Head(Meta(charset='utf-8'), Meta(name='viewport', content='width=device-width, initial-scale=1'),
        Style(registry.build_css() if css is None else css), Script(registry.build_js()), htmxsrc)
    return to_xml(Html(head, Body(*fts)))


For rendering in notebooks, we use an iframe to isolate the preview, so the styling of the notebook (e.g. solveit's CSS) can't affect the rendered markup, and vice versa. Its `srcdoc` attribute holds the complete document from `demo_html`. By default, the iframe automatically resizes to fit its contents, and we include a replay button so that we can reload dynamic components (e.g. ones that auto-dismiss).

In [ ]:
_sizer = Script(r"new ResizeObserver(() => frameElement.style.height = document.body.scrollHeight+24+'px').observe(document.body)")

def demo(
    *fts,         # Markup to render
    registry=reg, # Registry whose stylesheet and behavior script the preview carries
    css=None,     # Stylesheet for the document; None uses the registry's build_css()
    height=None,  # Fixed iframe height, e.g. '10rem', for top-layer content the frame can't measure; None follows the content
):
    "Render markup in an isolated, replayable notebook preview"
    if height is None: fts += (_sizer,)
    style = f'width:100%;box-sizing:border-box;border:1px solid #e3e3e8;border-radius:8px;height:{height or "auto"};'
    frame = Iframe(srcdoc=demo_html(*fts, registry=registry, css=css), title='answerui preview', style=style)
    replay = Button('↻', type='button', aria_label='replay', title='replay',
        onclick=r"const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc",
        style='border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px')
    return Div(Div(replay, style='text-align:right'), frame)


A preview does not require a registered component; Here `css=''` replaces the registry's stylesheet with an empty one, showing a native element exactly as the browser draws it:

In [ ]:
demo(Details(Summary('More information'), P('This disclosure is a native details element.')), css='')

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;&lt;/style&gt;
&lt;script&gt;(() =&gt; {
const rules = [];
const by = {};
rules.forEach(r =&gt; (by[r.ev + &#39;|&#39; + (r.cap ? 1 : 0)] ??= []).push(r));
for (const k in by) {
  const [ev, cap] = k.split(&#39;|&#39;);
  document.addEventListener(ev, e =&gt; {
    const t = e.target;
    if (!(t instanceof Element)) return;
    for (const r of by[k]) {
      if (r.deep) {
        if (t.matches(r.sel)) r.fn(t, e);
        t.querySelectorAll(r.sel).forEach(el =&gt; r.fn(el, e));
      } else {
        const el = t.closest(r.sel);
        if (el) r.fn(el, e);
      }
    }
  }, {capture: cap === &#39;1&#39;});
}
})();&lt;/script&gt;&lt;script src="https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js"&gt;&lt;/script&gt;  &lt;/head&gt;
  &lt;body&gt;
&lt;details&gt;&lt;summary&gt;More information&lt;/summary&gt;      &lt;p&gt;This disclosure is a native details element.&lt;/p&gt;
&lt;/details&gt;&lt;script&gt;new ResizeObserver(() =&gt; frameElement.style.height = document.body.scrollHeight+24+&#39;px&#39;).observe(document.body)&lt;/script&gt;  &lt;/body&gt;
&lt;/html&gt;
' title="answerui preview" style="width:100%;box-sizing:border-box;border:1px solid #e3e3e8;border-radius:8px;height:auto;"></iframe></div>

Let's recreate the notice from the earlier notebooks. Its styles and behaviors register together. The dismiss button uses an ordinary click rule; `autodismiss` starts a timer when htmx loads the notice.

In [ ]:
notice_css = r"""
notice-box {
  display: block;
  padding: .75rem 1rem;
  background: #f4f4f6;
  border: 1px solid #ccc;
  border-radius: 8px;
}
"""
NoticeBox = hypelt('NoticeBox', css=notice_css, layer='components', on={
    ('& [dismiss]', 'click'): r"el.closest('notice-box').remove()",
    ('&[autodismiss]', ':load'): r"await new Promise(r => setTimeout(r, +el.getAttribute('autodismiss') || 4000)); el.remove()"},
    doc='A dismissible notice.')

Click Dismiss, then the ↻ icon to restore the notice:

In [ ]:
demo(NoticeBox('Changes saved. ', Button('Dismiss', type='button', dismiss=True)))

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer components {
/* NoticeBox.base */
notice-box {
  display: block;
  padding: .75rem 1rem;
  background: #f4f4f6;
  border: 1px solid #ccc;
  border-radius: 8px;
}
}

@layer app {
/* NoticeBox.corners */
notice-box {
border-radius: 0;
}
}
&lt;/style&gt;
&lt;script&gt;(() =&gt; {
const rules = [{sel: ":is(notice-box) [dismiss]", ev: "click", cap: false, deep: false, async fn(el, e) { el.closest(&#39;notice-box&#39;).remove() }},
{sel: ":is(notice-box)[autodismiss]", ev: "htmx:load", cap: false, deep: true, async fn(el, e) { await new Promise(r =&gt; setTimeout(r, +el.getAttribute(&#39;autodismiss&#39;) || 4000)); el.remove() }}];
const by = {};
rules.forEach(r =&gt; (by[r.ev + &#39;|&#39; + (r.cap ? 1 : 0)] ??= []).push(r));
for (const k in by) {
  const [ev, cap] = k.split(&#39;|&#39;);
  document.addEventListener(ev, e =&gt; {
    const t = e.target;
    if (!(t instanceof Element)) return;
    for (const r of by[k]) {
      if (r.deep) {
        if (t.matches(r.sel)) r.fn(t, e);
        t.querySelectorAll(r.sel).forEach(el =&gt; r.fn(el, e));
      } else {
        const el = t.closest(r.sel);
        if (el) r.fn(el, e);
      }
    }
  }, {capture: cap === &#39;1&#39;});
}
})();&lt;/script&gt;&lt;script src="https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js"&gt;&lt;/script&gt;  &lt;/head&gt;
  &lt;body&gt;
&lt;notice-box&gt;Changes saved. &lt;button type="button" dismiss&gt;Dismiss&lt;/button&gt;&lt;/notice-box&gt;&lt;script&gt;new ResizeObserver(() =&gt; frameElement.style.height = document.body.scrollHeight+24+&#39;px&#39;).observe(document.body)&lt;/script&gt;  &lt;/body&gt;
&lt;/html&gt;
' title="answerui preview" style="width:100%;box-sizing:border-box;border:1px solid #e3e3e8;border-radius:8px;height:auto;"></iframe></div>

This notice disappears after four seconds:

In [ ]:
demo(NoticeBox('This notice disappears after four seconds.', autodismiss=4000), height='4rem')

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer components {
/* NoticeBox.base */
notice-box {
  display: block;
  padding: .75rem 1rem;
  background: #f4f4f6;
  border: 1px solid #ccc;
  border-radius: 8px;
}
}

@layer app {
/* NoticeBox.corners */
notice-box {
border-radius: 0;
}
}
&lt;/style&gt;
&lt;script&gt;(() =&gt; {
const rules = [{sel: ":is(notice-box) [dismiss]", ev: "click", cap: false, deep: false, async fn(el, e) { el.closest(&#39;notice-box&#39;).remove() }},
{sel: ":is(notice-box)[autodismiss]", ev: "htmx:load", cap: false, deep: true, async fn(el, e) { await new Promise(r =&gt; setTimeout(r, +el.getAttribute(&#39;autodismiss&#39;) || 4000)); el.remove() }}];
const by = {};
rules.forEach(r =&gt; (by[r.ev + &#39;|&#39; + (r.cap ? 1 : 0)] ??= []).push(r));
for (const k in by) {
  const [ev, cap] = k.split(&#39;|&#39;);
  document.addEventListener(ev, e =&gt; {
    const t = e.target;
    if (!(t instanceof Element)) return;
    for (const r of by[k]) {
      if (r.deep) {
        if (t.matches(r.sel)) r.fn(t, e);
        t.querySelectorAll(r.sel).forEach(el =&gt; r.fn(el, e));
      } else {
        const el = t.closest(r.sel);
        if (el) r.fn(el, e);
      }
    }
  }, {capture: cap === &#39;1&#39;});
}
})();&lt;/script&gt;&lt;script src="https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js"&gt;&lt;/script&gt;  &lt;/head&gt;
  &lt;body&gt;
&lt;notice-box autodismiss="4000"&gt;This notice disappears after four seconds.&lt;/notice-box&gt;  &lt;/body&gt;
&lt;/html&gt;
' title="answerui preview" style="width:100%;box-sizing:border-box;border:1px solid #e3e3e8;border-radius:8px;height:4rem;"></iframe></div>

A custom stylesheet lets us work on a candidate without changing the registry. Here we reuse the notice's CSS but replace its grey background with a pale yellow. The registered behaviors still apply.

In [ ]:
candidate_css = notice_css.replace('#f4f4f6', '#fff4cc')
demo(NoticeBox('Trying a warmer surface. ', Button('Dismiss', type='button', dismiss=True)), css=candidate_css)

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;
notice-box {
  display: block;
  padding: .75rem 1rem;
  background: #fff4cc;
  border: 1px solid #ccc;
  border-radius: 8px;
}
&lt;/style&gt;
&lt;script&gt;(() =&gt; {
const rules = [{sel: ":is(notice-box) [dismiss]", ev: "click", cap: false, deep: false, async fn(el, e) { el.closest(&#39;notice-box&#39;).remove() }},
{sel: ":is(notice-box)[autodismiss]", ev: "htmx:load", cap: false, deep: true, async fn(el, e) { await new Promise(r =&gt; setTimeout(r, +el.getAttribute(&#39;autodismiss&#39;) || 4000)); el.remove() }}];
const by = {};
rules.forEach(r =&gt; (by[r.ev + &#39;|&#39; + (r.cap ? 1 : 0)] ??= []).push(r));
for (const k in by) {
  const [ev, cap] = k.split(&#39;|&#39;);
  document.addEventListener(ev, e =&gt; {
    const t = e.target;
    if (!(t instanceof Element)) return;
    for (const r of by[k]) {
      if (r.deep) {
        if (t.matches(r.sel)) r.fn(t, e);
        t.querySelectorAll(r.sel).forEach(el =&gt; r.fn(el, e));
      } else {
        const el = t.closest(r.sel);
        if (el) r.fn(el, e);
      }
    }
  }, {capture: cap === &#39;1&#39;});
}
})();&lt;/script&gt;&lt;script src="https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js"&gt;&lt;/script&gt;  &lt;/head&gt;
  &lt;body&gt;
&lt;notice-box&gt;Trying a warmer surface. &lt;button type="button" dismiss&gt;Dismiss&lt;/button&gt;&lt;/notice-box&gt;&lt;script&gt;new ResizeObserver(() =&gt; frameElement.style.height = document.body.scrollHeight+24+&#39;px&#39;).observe(document.body)&lt;/script&gt;  &lt;/body&gt;
&lt;/html&gt;
' title="answerui preview" style="width:100%;box-sizing:border-box;border:1px solid #e3e3e8;border-radius:8px;height:auto;"></iframe></div>

An application override takes a different path. Registering CSS in the `app` layer makes it part of subsequent previews and application pages. This rule leaves the notice's definition and behavior alone:

In [ ]:
NoticeBox.rule('corners', 'border-radius: 1rem;', doc='Rounder notice corners.')
demo(NoticeBox('The current stylesheet includes the application override.'))


<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer components {
/* NoticeBox.base */
notice-box {
  display: block;
  padding: .75rem 1rem;
  background: #f4f4f6;
  border: 1px solid #ccc;
  border-radius: 8px;
}
}

@layer app {
/* NoticeBox.corners */
notice-box {
border-radius: 1rem;
}
}
&lt;/style&gt;
&lt;script&gt;(() =&gt; {
const rules = [{sel: ":is(notice-box) [dismiss]", ev: "click", cap: false, deep: false, async fn(el, e) { el.closest(&#39;notice-box&#39;).remove() }},
{sel: ":is(notice-box)[autodismiss]", ev: "htmx:load", cap: false, deep: true, async fn(el, e) { await new Promise(r =&gt; setTimeout(r, +el.getAttribute(&#39;autodismiss&#39;) || 4000)); el.remove() }}];
const by = {};
rules.forEach(r =&gt; (by[r.ev + &#39;|&#39; + (r.cap ? 1 : 0)] ??= []).push(r));
for (const k in by) {
  const [ev, cap] = k.split(&#39;|&#39;);
  document.addEventListener(ev, e =&gt; {
    const t = e.target;
    if (!(t instanceof Element)) return;
    for (const r of by[k]) {
      if (r.deep) {
        if (t.matches(r.sel)) r.fn(t, e);
        t.querySelectorAll(r.sel).forEach(el =&gt; r.fn(el, e));
      } else {
        const el = t.closest(r.sel);
        if (el) r.fn(el, e);
      }
    }
  }, {capture: cap === &#39;1&#39;});
}
})();&lt;/script&gt;&lt;script src="https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js"&gt;&lt;/script&gt;  &lt;/head&gt;
  &lt;body&gt;
&lt;notice-box&gt;The current stylesheet includes the application override.&lt;/notice-box&gt;&lt;script&gt;new ResizeObserver(() =&gt; frameElement.style.height = document.body.scrollHeight+24+&#39;px&#39;).observe(document.body)&lt;/script&gt;  &lt;/body&gt;
&lt;/html&gt;
' title="answerui preview" style="width:100%;box-sizing:border-box;border:1px solid #e3e3e8;border-radius:8px;height:auto;"></iframe></div>

## Reusable examples

A component can provide an example for someone exploring the library. We attach a no-argument markup function to the hyperelement with `example`, which stores it as `NoticeBox.example`, and `NoticeBox.demo()` renders that markup with the component's registry.

Attaching another example replaces the previous one; several variants can be composed in one function.

In [ ]:
def example(h):
    "Decorate a no-argument markup function as `h.example`, the markup `h.demo()` renders"
    def _attach(f):
        h.example = f
        return f
    return _attach

@patch
def demo(
    self:Hyperelement,
    css=None,    # Stylesheet for the document; None uses the registry's build_css()
    height=None, # Fixed iframe height; None follows the content
):
    "Render this hyperelement's example with its registry's current styles and behaviors"
    if not hasattr(self, 'example'): raise ValueError(f'{self.__name__} has no example; attach one with @example({self.__name__})')
    return demo(self.example(), registry=self.registry, css=css, height=height)


In [ ]:
@example(NoticeBox)
def notice_example():
    return Div(NoticeBox('Changes saved. ', Button('Dismiss', type='button', dismiss=True)),
        NoticeBox('This notice stays visible.'), style='display:grid;gap:1rem')

The example itself is ordinary markup:

In [ ]:
to_xml(NoticeBox.example())

'<div style="display:grid;gap:1rem">\n<notice-box>Changes saved. <button type="button" dismiss>Dismiss</button></notice-box><notice-box>This notice stays visible.</notice-box></div>\n'

Rendered with the current assets:

In [ ]:
NoticeBox.demo()


<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer components {
/* NoticeBox.base */
notice-box {
  display: block;
  padding: .75rem 1rem;
  background: #f4f4f6;
  border: 1px solid #ccc;
  border-radius: 8px;
}
}

@layer app {
/* NoticeBox.corners */
notice-box {
border-radius: 1rem;
}
}
&lt;/style&gt;
&lt;script&gt;(() =&gt; {
const rules = [{sel: ":is(notice-box) [dismiss]", ev: "click", cap: false, deep: false, async fn(el, e) { el.closest(&#39;notice-box&#39;).remove() }},
{sel: ":is(notice-box)[autodismiss]", ev: "htmx:load", cap: false, deep: true, async fn(el, e) { await new Promise(r =&gt; setTimeout(r, +el.getAttribute(&#39;autodismiss&#39;) || 4000)); el.remove() }}];
const by = {};
rules.forEach(r =&gt; (by[r.ev + &#39;|&#39; + (r.cap ? 1 : 0)] ??= []).push(r));
for (const k in by) {
  const [ev, cap] = k.split(&#39;|&#39;);
  document.addEventListener(ev, e =&gt; {
    const t = e.target;
    if (!(t instanceof Element)) return;
    for (const r of by[k]) {
      if (r.deep) {
        if (t.matches(r.sel)) r.fn(t, e);
        t.querySelectorAll(r.sel).forEach(el =&gt; r.fn(el, e));
      } else {
        const el = t.closest(r.sel);
        if (el) r.fn(el, e);
      }
    }
  }, {capture: cap === &#39;1&#39;});
}
})();&lt;/script&gt;&lt;script src="https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js"&gt;&lt;/script&gt;  &lt;/head&gt;
  &lt;body&gt;
    &lt;div style="display:grid;gap:1rem"&gt;
&lt;notice-box&gt;Changes saved. &lt;button type="button" dismiss&gt;Dismiss&lt;/button&gt;&lt;/notice-box&gt;&lt;notice-box&gt;This notice stays visible.&lt;/notice-box&gt;    &lt;/div&gt;
&lt;script&gt;new ResizeObserver(() =&gt; frameElement.style.height = document.body.scrollHeight+24+&#39;px&#39;).observe(document.body)&lt;/script&gt;  &lt;/body&gt;
&lt;/html&gt;
' title="answerui preview" style="width:100%;box-sizing:border-box;border:1px solid #e3e3e8;border-radius:8px;height:auto;"></iframe></div>

`NoticeBox.example()` is fresh markup, ordinary FastHTML that can be inspected or reused in a larger composition, and `NoticeBox.demo()` takes the preview options such as `height` and `css`. Each call builds a new document from the current example, styles, and behaviors. The replay button does something different: it reloads the preview's existing snapshot, restarting its interactions without consulting Python.

For a shipped component, the decorated example function is exported in its component notebook, so importing the module attaches the example without rendering anything.


A customization, then another preview. The rule is registered, so the new document carries it:

In [ ]:
NoticeBox.rule('corners', 'border-radius: 0;', doc='Square notice corners.')
NoticeBox.demo()

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer components {
/* NoticeBox.base */
notice-box {
  display: block;
  padding: .75rem 1rem;
  background: #f4f4f6;
  border: 1px solid #ccc;
  border-radius: 8px;
}
}

@layer app {
/* NoticeBox.corners */
notice-box {
border-radius: 0;
}
}
&lt;/style&gt;
&lt;script&gt;(() =&gt; {
const rules = [{sel: ":is(notice-box) [dismiss]", ev: "click", cap: false, deep: false, async fn(el, e) { el.closest(&#39;notice-box&#39;).remove() }},
{sel: ":is(notice-box)[autodismiss]", ev: "htmx:load", cap: false, deep: true, async fn(el, e) { await new Promise(r =&gt; setTimeout(r, +el.getAttribute(&#39;autodismiss&#39;) || 4000)); el.remove() }}];
const by = {};
rules.forEach(r =&gt; (by[r.ev + &#39;|&#39; + (r.cap ? 1 : 0)] ??= []).push(r));
for (const k in by) {
  const [ev, cap] = k.split(&#39;|&#39;);
  document.addEventListener(ev, e =&gt; {
    const t = e.target;
    if (!(t instanceof Element)) return;
    for (const r of by[k]) {
      if (r.deep) {
        if (t.matches(r.sel)) r.fn(t, e);
        t.querySelectorAll(r.sel).forEach(el =&gt; r.fn(el, e));
      } else {
        const el = t.closest(r.sel);
        if (el) r.fn(el, e);
      }
    }
  }, {capture: cap === &#39;1&#39;});
}
})();&lt;/script&gt;&lt;script src="https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js"&gt;&lt;/script&gt;  &lt;/head&gt;
  &lt;body&gt;
    &lt;div style="display:grid;gap:1rem"&gt;
&lt;notice-box&gt;Changes saved. &lt;button type="button" dismiss&gt;Dismiss&lt;/button&gt;&lt;/notice-box&gt;&lt;notice-box&gt;This notice stays visible.&lt;/notice-box&gt;    &lt;/div&gt;
&lt;script&gt;new ResizeObserver(() =&gt; frameElement.style.height = document.body.scrollHeight+24+&#39;px&#39;).observe(document.body)&lt;/script&gt;  &lt;/body&gt;
&lt;/html&gt;
' title="answerui preview" style="width:100%;box-sizing:border-box;border:1px solid #e3e3e8;border-radius:8px;height:auto;"></iframe></div>

Redefining the component keeps the object, and its example with it:

In [ ]:
NoticeBox = hypelt('NoticeBox', css=notice_css, layer='components', doc='A persistent notice.')
test_is(NoticeBox.example, notice_example)
NoticeBox.demo()

<div><div style="text-align:right"><button type="button" aria-label="replay" onclick="const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc" title="replay" style="border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px">↻</button></div><iframe srcdoc='&lt;!doctype html&gt;
&lt;html&gt;
  &lt;head&gt;
    &lt;meta charset="utf-8"&gt;
    &lt;meta name="viewport" content="width=device-width, initial-scale=1"&gt;
    &lt;style&gt;@layer reset, tokens, base, layout, components, app, utilities;

@layer components {
/* NoticeBox.base */
notice-box {
  display: block;
  padding: .75rem 1rem;
  background: #f4f4f6;
  border: 1px solid #ccc;
  border-radius: 8px;
}
}

@layer app {
/* NoticeBox.corners */
notice-box {
border-radius: 0;
}
}
&lt;/style&gt;
&lt;script&gt;(() =&gt; {
const rules = [];
const by = {};
rules.forEach(r =&gt; (by[r.ev + &#39;|&#39; + (r.cap ? 1 : 0)] ??= []).push(r));
for (const k in by) {
  const [ev, cap] = k.split(&#39;|&#39;);
  document.addEventListener(ev, e =&gt; {
    const t = e.target;
    if (!(t instanceof Element)) return;
    for (const r of by[k]) {
      if (r.deep) {
        if (t.matches(r.sel)) r.fn(t, e);
        t.querySelectorAll(r.sel).forEach(el =&gt; r.fn(el, e));
      } else {
        const el = t.closest(r.sel);
        if (el) r.fn(el, e);
      }
    }
  }, {capture: cap === &#39;1&#39;});
}
})();&lt;/script&gt;&lt;script src="https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js"&gt;&lt;/script&gt;  &lt;/head&gt;
  &lt;body&gt;
    &lt;div style="display:grid;gap:1rem"&gt;
&lt;notice-box&gt;Changes saved. &lt;button type="button" dismiss&gt;Dismiss&lt;/button&gt;&lt;/notice-box&gt;&lt;notice-box&gt;This notice stays visible.&lt;/notice-box&gt;    &lt;/div&gt;
&lt;script&gt;new ResizeObserver(() =&gt; frameElement.style.height = document.body.scrollHeight+24+&#39;px&#39;).observe(document.body)&lt;/script&gt;  &lt;/body&gt;
&lt;/html&gt;
' title="answerui preview" style="width:100%;box-sizing:border-box;border:1px solid #e3e3e8;border-radius:8px;height:auto;"></iframe></div>

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()